# Web Analytics Customer Segmentation using K-Means\n\nThis notebook segments online customers using website behaviour, ecommerce activity, and engagement metrics.\n

In [ ]:
import pandas as pd\nimport matplotlib.pyplot as plt\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.cluster import KMeans\nfrom sklearn.decomposition import PCA\nfrom sklearn.metrics import silhouette_score\n\ndf = pd.read_csv('../data/web_customer_analytics_6000.csv')\ndf.head()\n

In [ ]:
features = ['sessions','pageviews','avg_session_duration_sec','bounce_rate','product_views','cart_additions','transactions','revenue_gbp','days_since_last_visit','email_clicks','support_visits','discount_usage_rate','returning_user','conversion_rate','cart_to_purchase_rate','pages_per_session','engagement_score']\nX = StandardScaler().fit_transform(df[features])\n

In [ ]:
results = []\nfor k in range(2, 8):\n    model = KMeans(n_clusters=k, random_state=42, n_init=5, max_iter=150)\n    labels = model.fit_predict(X)\n    results.append({'k': k, 'inertia': model.inertia_, 'silhouette_score': silhouette_score(X, labels, sample_size=500, random_state=42)})\nk_results = pd.DataFrame(results)\nk_results\n

In [ ]:
model = KMeans(n_clusters=5, random_state=42, n_init=10, max_iter=200)\ndf['segment'] = model.fit_predict(X)\nsegment_profile = df.groupby('segment').agg(customers=('customer_id','count'), avg_sessions=('sessions','mean'), avg_revenue_gbp=('revenue_gbp','mean'), avg_conversion_rate=('conversion_rate','mean'), avg_engagement_score=('engagement_score','mean')).round(2)\nsegment_profile\n

In [ ]:
components = PCA(n_components=2, random_state=42).fit_transform(X)\nplt.figure(figsize=(9,6))\nfor segment in sorted(df['segment'].unique()):\n    temp = components[df['segment'].values == segment]\n    plt.scatter(temp[:,0], temp[:,1], s=10, alpha=.5, label=f'Segment {segment}')\nplt.title('Customer Segments Visualized with PCA')\nplt.legend()\nplt.show()\n

## Business Recommendations\n\n- High value loyal customers: loyalty benefits and premium bundles.\n- Product researchers: reviews, comparison content, and remarketing.\n- Deal-driven converters: margin-aware discounts.\n- Engaged non-buyers: checkout UX and basket recovery.\n- Low engagement visitors: low-cost reactivation and retargeting suppression.\n